In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
get_ipython().system('pip install seqeval')

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "seqeval", "scikit-learn", "nltk"])

In [ ]:
import pandas as pd
import numpy as np
import ast, re, random, warnings, time, os, json
from collections import Counter, defaultdict
from copy import deepcopy

from seqeval.metrics import (
    f1_score as seqeval_f1,
    precision_score as seqeval_p,
    recall_score as seqeval_r,
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForTokenClassification,
    get_linear_schedule_with_warmup
)
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('english'))

os.environ["HF_TOKEN"] = ""
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)


In [ ]:
FEW_SHOT_FRACTION = 0.02      # <-- GANTI: 0.02, 0.05, atau 0.10
FEW_SHOT_LABEL    = '2%'     # <-- GANTI: '2%', '5%', atau '10%'
# =============================================

MODEL_NAME    = 'dmis-lab/biobert-v1.1'
NUM_EPOCHS    = 5
LEARNING_RATE = 5e-5
BATCH_SIZE    = 8
MAX_SEQ_LEN   = 512

# BC5CDR: Chemical only (Disease diabaikan, sesuai paper Bartolini Table 2)
TAG2IDX = {
    'O': 0,
    'B-Chemical': 1,
    'I-Chemical': 2,
}
IDX2TAG = {v: k for k, v in TAG2IDX.items()}
NUM_TAGS = len(TAG2IDX)

# BC5CDR raw CSV: 0=O, 1=B-Chemical, 2=B-Disease, 3=I-Disease, 4=I-Chemical
# Disease tags (2,3) diubah jadi O
CSV_TAG_TO_IOB = {
    0: 'O',
    1: 'B-Chemical',
    2: 'O',            # B-Disease -> O (diabaikan)
    3: 'O',            # I-Disease -> O (diabaikan)
    4: 'I-Chemical',
}

# [R2C2] Five training seeds for mean ± SD reporting
SEEDS = [64, 128, 256, 512, 1024]
# [R2C2] Fixed data splitting seed for reproducibility across all corpora and settings
SAMPLING_SEED = 64

RS_DEFAULT   = 15
AUG_TOP_K    = 5

OUTPUT_DIR = '/kaggle/working/hasil_bc5cdr_' + FEW_SHOT_LABEL.replace('%', 'pct')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Dataset: BC5CDR")
print(f"Scenario: {FEW_SHOT_LABEL}")
print(f"Tags: {list(TAG2IDX.keys())}")
print(f"Seeds: {SEEDS}")
print(f"Epochs: {NUM_EPOCHS}, LR: {LEARNING_RATE}, Batch: {BATCH_SIZE}, MaxLen: {MAX_SEQ_LEN}")
print(f"Rs={RS_DEFAULT}, IR=Adaptive(median/Q3), TopK={AUG_TOP_K}")

In [ ]:
# =============================================
#   PARSER KHUSUS BC5CDR
# =============================================
def parse_bc5cdr(filepath):
    """
    Parse BC5CDR CSV format using csv.reader (handles multi-line quoted fields).
      tokens,tags
      "['Naloxone' 'reverses' ...]",[1 0 0 ...]
    """
    import csv
    sentences = []
    sent_id = 0
    
    with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
        reader = csv.reader(f)
        header = next(reader)  # skip header
        
        for row in reader:
            if len(row) < 2:
                continue
            tokens_str = row[0].strip()
            tags_str = row[1].strip()
            
            # Parse tokens: ['word1' 'word2' ...] -> extract words in single quotes
            tokens = re.findall(r"'([^']*)'", tokens_str)
            if not tokens:
                tokens = re.findall(r'"([^"]*)"', tokens_str)
            
            # Parse tags: [0 1 2 ...] -> extract integers
            tags_clean = tags_str.strip('[]')
            tags_raw = [int(t) for t in tags_clean.split()]
            
            # Convert to IOB tags (Disease tags -> O)
            iob_tags = [CSV_TAG_TO_IOB.get(t, 'O') for t in tags_raw]
            
            if len(tokens) == len(iob_tags) and len(tokens) > 0:
                sentences.append({
                    'id': 'bc5cdr_' + str(sent_id),
                    'tokens': tokens,
                    'tags': iob_tags,
                })
                sent_id += 1
    
    return sentences

# === CELL BREAK ===
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# [R1C7] Saves split indices to JSON for reproducibility verification
def sample_few_shot(sentences, fraction, seed=SAMPLING_SEED, save_path=None):
    rng = random.Random(seed)
    n = max(1, int(len(sentences) * fraction))
    indices = list(range(len(sentences)))
    rng.shuffle(indices)
    selected_indices = indices[:n]
    if save_path:
        with open(save_path, 'w') as f:
            json.dump({'seed': seed, 'fraction': fraction, 'n': n, 'indices': selected_indices}, f)
    return [sentences[i] for i in selected_indices]

# === CELL BREAK ===
def is_valid_token(token):
    return token.lower() not in STOPWORDS and not (len(token) == 1 and token.isalnum())

def ibus_undersample_sentence(sent, rs):
    tokens, tags = sent['tokens'], sent['tags']
    n_pos = sum(1 for t in tags if t != 'O')
    n_neg = sum(1 for t in tags if t == 'O')
    if n_pos == 0: return None
    if n_neg / n_pos <= rs: return deepcopy(sent)
    maj_req = int(rs * n_pos)
    selected = [t != 'O' for t in tags]
    n_sel = 0
    entities = []
    i = 0
    while i < len(tags):
        if tags[i] != 'O':
            start = i
            while i < len(tags) and tags[i] != 'O': i += 1
            entities.append((start, i))
        else: i += 1

    def select_adj(check_valid=True):
        nonlocal n_sel
        for start, end in entities:
            if n_sel >= maj_req: return
            il = start - 1
            while il >= 0:
                if not selected[il] and tags[il] == 'O':
                    if not check_valid or is_valid_token(tokens[il]):
                        selected[il] = True; n_sel += 1
                    break
                il -= 1
            if n_sel >= maj_req: return
            ir_idx = end
            while ir_idx < len(tokens):
                if not selected[ir_idx] and tags[ir_idx] == 'O':
                    if not check_valid or is_valid_token(tokens[ir_idx]):
                        selected[ir_idx] = True; n_sel += 1
                    break
                ir_idx += 1

    n_inv = sum(1 for i, t in enumerate(tags) if t == 'O' and not is_valid_token(tokens[i]))
    if maj_req <= (n_neg - n_inv):
        while n_sel < maj_req:
            prev = n_sel; select_adj(True)
            if n_sel == prev: break
        for i in range(len(tokens)):
            if n_sel >= maj_req: break
            if not selected[i] and tags[i] == 'O' and is_valid_token(tokens[i]):
                selected[i] = True; n_sel += 1
    else:
        while n_sel < min(maj_req, n_neg - n_inv):
            prev = n_sel; select_adj(True)
            if n_sel == prev: break
        for i in range(len(tokens)):
            if n_sel >= maj_req: break
            if not selected[i] and tags[i] == 'O' and is_valid_token(tokens[i]):
                selected[i] = True; n_sel += 1
        while n_sel < maj_req:
            prev = n_sel; select_adj(False)
            if n_sel == prev: break
        for i in range(len(tokens)):
            if n_sel >= maj_req: break
            if not selected[i] and tags[i] == 'O':
                selected[i] = True; n_sel += 1

    return {
        'id': sent['id'],
        'tokens': [t for t, s in zip(tokens, selected) if s],
        'tags':   [t for t, s in zip(tags, selected) if s]
    }

# [R1C7] Saves iBUS filtering log (original/kept/removed tokens per sentence)
def ibus_undersample_dataset(sentences, rs, log_path=None):
    result = []
    log_rows = []
    for sent in sentences:
        orig_len = len(sent['tokens'])
        us = ibus_undersample_sentence(sent, rs)
        if us is not None:
            result.append(us)
            kept_len = len(us['tokens'])
        else:
            result.append(deepcopy(sent))
            kept_len = orig_len
        log_rows.append({'id': sent['id'], 'original_tokens': orig_len,
                         'kept_tokens': kept_len, 'removed': orig_len - kept_len})
    print(f"    iBUS (Rs={rs}): {len(sentences)} -> {len(result)} sentences")
    if log_path:
        pd.DataFrame(log_rows).to_csv(log_path, index=False)
    return result

# === CELL BREAK ===
def extract_mentions_from_sentences(sentences):
    mention_contexts = defaultdict(list)
    mention_types = {}
    for sent in sentences:
        tokens, tags = sent['tokens'], sent['tags']
        i = 0
        while i < len(tags):
            if tags[i].startswith('B-'):
                start, etype = i, tags[i][2:]
                i += 1
                while i < len(tags) and tags[i].startswith('I-'): i += 1
                mention = " ".join(tokens[start:i])
                mention_contexts[mention].append((tokens, start, i))
                mention_types[mention] = etype
            else: i += 1
    return mention_contexts, mention_types

def build_entity_embeddings(mention_contexts, model_name=MODEL_NAME):
    emb_device = "cuda" if torch.cuda.is_available() else "cpu"
    emb_tokenizer = AutoTokenizer.from_pretrained(model_name)
    emb_model = AutoModel.from_pretrained(model_name).to(emb_device)
    emb_model.eval()
    embeddings = {}
    for mention, contexts in mention_contexts.items():
        c = len(contexts); lr = 1.0 / c; v_concept = None
        for tokens, start, end in contexts:
            sentence = " ".join(tokens)
            inputs = emb_tokenizer(sentence, return_tensors="pt", truncation=True, max_length=512)
            inputs = {k: v.to(emb_device) for k, v in inputs.items()}
            with torch.no_grad():
                hidden = emb_model(**inputs).last_hidden_state[0]
            v_context = hidden[1:-1].mean(dim=0).cpu().numpy()
            if v_concept is None: v_concept = v_context.copy()
            else:
                norm_prod = np.linalg.norm(v_concept) * np.linalg.norm(v_context) + 1e-8
                sim = max(0, np.dot(v_concept, v_context) / norm_prod)
                v_concept = v_concept + lr * (1 - sim) * v_context
        embeddings[mention] = v_concept
    del emb_model; torch.cuda.empty_cache()
    return embeddings

def compute_similarity_rankings(embeddings, mention_types):
    mentions = list(embeddings.keys())
    if len(mentions) < 2: return {}
    emb_matrix = np.array([embeddings[m] for m in mentions])
    sim_mat = cosine_similarity(emb_matrix)
    rankings = {}
    for i, mention in enumerate(mentions):
        mtype = mention_types.get(mention, "")
        scores = [(mentions[j], sim_mat[i][j])
                  for j in range(len(mentions))
                  if i != j and mention_types.get(mentions[j], "") == mtype]
        scores.sort(key=lambda x: x[1], reverse=True)
        rankings[mention] = scores
    return rankings

def augment_sentence(sent, rankings, k):
    tokens, tags = sent['tokens'], sent['tags']
    mentions = []
    i = 0
    while i < len(tags):
        if tags[i].startswith('B-'):
            start = i; etype = tags[i][2:]; i += 1
            while i < len(tags) and tags[i].startswith('I-'): i += 1
            mentions.append((start, i, " ".join(tokens[start:i]), etype))
        else: i += 1
    if not mentions: return []
    augmented = []
    for aug_idx in range(k):
        # Build the augmented sentence left-to-right so that token offsets never
        # need to be recomputed. Each original mention is either replaced by its
        # aug_idx-th ranked candidate or copied verbatim; the non-entity spans
        # between mentions are always copied verbatim.
        new_tokens, new_tags = [], []
        cursor = 0
        n_rep = 0
        for (start, end, mention_text, etype) in mentions:
            # copy the non-entity tokens that precede this mention
            new_tokens.extend(tokens[cursor:start])
            new_tags.extend(tags[cursor:start])
            sims = rankings.get(mention_text)
            if sims is not None and aug_idx < len(sims):
                replacement, _ = sims[aug_idx]
                rep_tok = replacement.split()
                rep_tags = ['B-' + etype] + ['I-' + etype] * (len(rep_tok) - 1)
                new_tokens.extend(rep_tok)
                new_tags.extend(rep_tags)
                n_rep += 1
            else:
                # no candidate available: keep the original mention unchanged
                new_tokens.extend(tokens[start:end])
                new_tags.extend(tags[start:end])
            cursor = end
        # copy the trailing non-entity tokens after the last mention
        new_tokens.extend(tokens[cursor:])
        new_tags.extend(tags[cursor:])
        if n_rep > 0:
            augmented.append({'id': str(sent['id']) + '_aug' + str(aug_idx), 'tokens': new_tokens, 'tags': new_tags})
    return augmented

def compute_ir_stats(sentences):
    """Compute IR distribution stats for adaptive thresholding."""
    irs = []
    for sent in sentences:
        n_pos = sum(1 for t in sent['tags'] if t != 'O')
        if n_pos == 0: continue
        n_neg = sum(1 for t in sent['tags'] if t == 'O')
        irs.append(n_neg / n_pos)
    if not irs:
        return 1.0, 2.0
    median_ir = float(np.median(irs))
    q3_ir = float(np.percentile(irs, 75))
    return median_ir, q3_ir

def run_cosiner(sentences, rankings, imbalance_aware=False, ir_median=None, ir_q3=None):
    all_aug = []
    for sent in sentences:
        tags = sent['tags']
        n_pos = sum(1 for t in tags if t != 'O')
        if n_pos == 0: continue
        n_neg = sum(1 for t in tags if t == 'O')
        ir = n_neg / n_pos
        if imbalance_aware:
            if ir <= ir_median: continue
            scale = min(1.0, (ir - ir_median) / (ir_q3 - ir_median + 1e-8))
            k = max(1, round(AUG_TOP_K * scale))
        else:
            k = AUG_TOP_K
        augs = augment_sentence(sent, rankings, k)
        all_aug.extend(augs)
    mode = "ADAMER" if imbalance_aware else "COSINER"
    print(f"    {mode}: generated {len(all_aug)} augmented sentences")
    return all_aug

# === CELL BREAK ===
class NERDatasetBERT(Dataset):
    def __init__(self, sentences, tokenizer, tag2idx, max_len=MAX_SEQ_LEN):
        self.sentences = sentences
        self.tokenizer = tokenizer
        self.tag2idx = tag2idx
        self.max_len = max_len
    def __len__(self): return len(self.sentences)
    def __getitem__(self, idx):
        sent = self.sentences[idx]
        tokens, tags = sent['tokens'], sent['tags']
        encoding = self.tokenizer(tokens, is_split_into_words=True,
            max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        word_ids = encoding.word_ids()
        aligned = []; prev = None
        for wid in word_ids:
            if wid is None: aligned.append(-100)
            elif wid != prev:
                aligned.append(self.tag2idx[tags[wid]] if wid < len(tags) else -100)
            else:
                if wid < len(tags):
                    tag = tags[wid]
                    if tag.startswith('B-'): aligned.append(self.tag2idx['I-' + tag[2:]])
                    else: aligned.append(self.tag2idx[tag])
                else: aligned.append(-100)
            prev = wid
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(aligned, dtype=torch.long)
        }
fs=FEW_SHOT_FRACTION
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# [R2C5] Compute training-set statistics for Table 8 (N_sentences, entity/non-entity tokens, Mean_IR, Aug_instances)
def compute_dataset_stats(sentences, config_label, aug_count=0):
    """Compute training-set statistics for Table 8 (R2C5)."""
    n_sent = len(sentences)
    entity_tok     = sum(1 for s in sentences for t in s['tags'] if t != 'O')
    non_entity_tok = sum(1 for s in sentences for t in s['tags'] if t == 'O')
    irs = [sum(1 for t in s['tags'] if t == 'O') / sum(1 for t in s['tags'] if t != 'O')
           for s in sentences if sum(1 for t in s['tags'] if t != 'O') > 0]
    mean_ir = round(sum(irs) / len(irs), 2) if irs else 0.0
    return {'Config': config_label, 'N_sentences': n_sent,
            'Entity_tokens': entity_tok, 'Non_entity_tokens': non_entity_tok,
            'Mean_IR': mean_ir, 'Aug_instances': aug_count}

def evaluate_model(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels']
            preds = model(input_ids=ids, attention_mask=mask).logits.argmax(dim=-1).cpu().tolist()
            for ps, ls in zip(preds, labels.tolist()):
                pt, tt = [], []
                for p, l in zip(ps, ls):
                    if l != -100:
                        pt.append(IDX2TAG.get(p, 'O'))
                        tt.append(IDX2TAG.get(l, 'O'))
                if tt: y_true.append(tt); y_pred.append(pt)
    return seqeval_f1(y_true, y_pred), seqeval_p(y_true, y_pred), seqeval_r(y_true, y_pred)

# [R2C6] Returns per-token predictions for failure mode analysis (FN, FP, boundary errors)
def evaluate_model_with_preds(model, loader):
    """Same as evaluate_model but returns per-sentence predictions (R2C6)."""
    model.eval()
    y_true, y_pred, sent_preds = [], [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels']
            preds = model(input_ids=ids, attention_mask=mask).logits.argmax(dim=-1).cpu().tolist()
            for ps, ls in zip(preds, labels.tolist()):
                pt, tt = [], []
                for p, l in zip(ps, ls):
                    if l != -100:
                        pt.append(IDX2TAG.get(p, 'O'))
                        tt.append(IDX2TAG.get(l, 'O'))
                if tt:
                    y_true.append(tt); y_pred.append(pt)
                    sent_preds.append({'true': tt, 'pred': pt})
    f1 = seqeval_f1(y_true, y_pred)
    p  = seqeval_p(y_true, y_pred)
    r  = seqeval_r(y_true, y_pred)
    return f1, p, r, sent_preds

def train_and_evaluate(train_sents, val_sents, test_sents, seed, name="", silent=False):
    set_seed(seed)
    train_ds = NERDatasetBERT(train_sents, tokenizer, TAG2IDX)
    val_ds   = NERDatasetBERT(val_sents, tokenizer, TAG2IDX)
    test_ds  = NERDatasetBERT(test_sents, tokenizer, TAG2IDX)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE*4, shuffle=False)
    test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE*4, shuffle=False)

    model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, num_labels=NUM_TAGS).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    scheduler = get_linear_schedule_with_warmup(optimizer, 0, len(train_loader) * NUM_EPOCHS)

    best_f1, best_state = 0, None
    t0 = time.time()
    for epoch in range(NUM_EPOCHS):
        model.train(); total_loss = 0
        for batch in train_loader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            loss = model(input_ids=ids, attention_mask=mask, labels=labels).loss
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        vf1, vp, vr = evaluate_model(model, val_loader)
        if not silent:
            print(f"    [{name}] Epoch {epoch+1}/{NUM_EPOCHS}: loss={avg_loss:.4f}, val_F1={vf1*100:.2f}%")
        if vf1 > best_f1:
            best_f1 = vf1; best_state = deepcopy(model.state_dict())

    if best_state: model.load_state_dict(best_state)
    tf1, tp, tr = evaluate_model(model, test_loader)
    elapsed = time.time() - t0
    if not silent:
        print(f"    [{name}] Test: F1={tf1*100:.2f}%, P={tp*100:.2f}%, R={tr*100:.2f}%, time={elapsed:.1f}s")
    del model; torch.cuda.empty_cache()
    return {'f1': tf1, 'p': tp, 'r': tr, 'time': elapsed}

def run_multi_seed(train_sents, val_sents, test_sents, name="", save_preds_path=None):
    runs = []
    for i, seed in enumerate(SEEDS):
        print(f"\n  Seed {seed}:")
        r = train_and_evaluate(train_sents, val_sents, test_sents, seed, name)
        runs.append(r)
        if save_preds_path and i == len(SEEDS) - 1:
            set_seed(seed)
            test_ds = NERDatasetBERT(test_sents, tokenizer, TAG2IDX)
            test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE*4, shuffle=False)
            model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, num_labels=NUM_TAGS).to(device)
            _, _, _, sent_preds = evaluate_model_with_preds(model, test_loader)
            pred_rows = []
            for sid, sp in enumerate(sent_preds):
                for tidx, (t, p) in enumerate(zip(sp['true'], sp['pred'])):
                    error_type = 'correct' if t == p else (
                        'FN' if t != 'O' and p == 'O' else (
                        'FP' if t == 'O' and p != 'O' else 'boundary'))
                    pred_rows.append({'sent_id': sid, 'tok_idx': tidx,
                                      'true': t, 'pred': p, 'error_type': error_type})
            pd.DataFrame(pred_rows).to_csv(save_preds_path + '.csv', index=False)
            print(f"    Predictions saved: {save_preds_path}.csv")
            del model; torch.cuda.empty_cache()
    f1s = np.array([r['f1'] for r in runs])
    ps  = np.array([r['p'] for r in runs])
    rs  = np.array([r['r'] for r in runs])
    ci95 = 1.96 * f1s.std(ddof=1) / np.sqrt(len(f1s)) if len(f1s) > 1 else 0.0
    result = {
        'f1_mean': f1s.mean(), 'f1_std': f1s.std(ddof=1) if len(f1s) > 1 else 0.0, 'f1_ci95': ci95,
        'p_mean': ps.mean(), 'p_std': ps.std(ddof=1) if len(ps) > 1 else 0.0,
        'r_mean': rs.mean(), 'r_std': rs.std(ddof=1) if len(rs) > 1 else 0.0,
        'f1_scores': f1s.tolist(), 'p_scores': ps.tolist(), 'r_scores': rs.tolist(),
        'avg_time': np.mean([r['time'] for r in runs]),
        'n_train': len(train_sents),
    }
    print(f"\n  >> {name}: F1={f1s.mean()*100:.2f}% +/- {result['f1_std']:.3f} (CI: +/-{ci95:.3f})")
    return result

In [ ]:
# ============================================================================
# STATISTICAL SIGNIFICANCE TEST — All-Pairwise (paired by seed)
# ----------------------------------------------------------------------------
# - Shapiro-Wilk on paired DIFFERENCES (not on marginal distributions)
# - Paired t-test if differences are normal, else Wilcoxon signed-rank
# - Effect size: Cohen's d_z (parametric) OR matched-pairs rank-biserial r
# - Multiple comparison correction: Bonferroni and Benjamini-Hochberg FDR
# - 95% CI of the mean difference (Student t-interval)
# ============================================================================
from scipy import stats as _scistats
from itertools import combinations as _combinations

_ALPHA = 0.05

def _bh_fdr(pvals):
    """Benjamini-Hochberg step-up FDR-adjusted p-values."""
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    if n == 0:
        return pvals
    order = np.argsort(pvals)
    ranked = pvals[order]
    adj = np.empty(n)
    prev = 1.0
    for k in range(n - 1, -1, -1):
        rank = k + 1
        val = min(prev, ranked[k] * n / rank)
        prev = val
        adj[order[k]] = val
    return np.minimum(adj, 1.0)

def _rank_biserial(diff):
    """Matched-pairs rank-biserial correlation (effect size for Wilcoxon)."""
    diff = np.asarray(diff, dtype=float)
    diff = diff[diff != 0]
    if len(diff) == 0:
        return 0.0
    ranks = _scistats.rankdata(np.abs(diff))
    w_pos = ranks[diff > 0].sum()
    w_neg = ranks[diff < 0].sum()
    total = w_pos + w_neg
    return float((w_pos - w_neg) / total) if total > 0 else 0.0

def _magnitude(value, parametric):
    """Map a numeric effect size to a qualitative magnitude label."""
    v = abs(value)
    if parametric:
        # Cohen's d thresholds
        return "Large" if v >= 0.8 else "Medium" if v >= 0.5 else "Small" if v >= 0.2 else "Negligible"
    # rank-biserial r thresholds
    return "Large" if v >= 0.5 else "Medium" if v >= 0.3 else "Small" if v >= 0.1 else "Negligible"

def run_pairwise_significance(results_dict, dataset, scenario, run_label, output_dir, alpha=_ALPHA):
    """
    All-pairwise paired significance test on per-seed F1 scores.

    Parameters
    ----------
    results_dict : dict  -> {method_name: {'f1_scores': [F1_per_seed in 0..1]}}
    dataset      : str   -> e.g. 'BC2GM', 'BC5CDR', 'NCBI'
    scenario     : str   -> e.g. '2%', '5%', '10%'
    run_label    : str   -> e.g. 'run1', 'run2'
    output_dir   : str   -> directory in which to save the CSV
    alpha        : float -> significance level (default 0.05)

    Saves   : significance_<dataset>_<run_label>_<scenario>.csv
    Returns : pandas.DataFrame of pairwise results.
    """
    methods_list = list(results_dict.keys())
    _run_field = run_label if run_label else 'full'
    f1_matrix = {m: np.asarray(results_dict[m]['f1_scores'], dtype=float)
                 for m in methods_list}
    pairs = list(_combinations(methods_list, 2))
    rows, raw_pvals = [], []

    for a, b in pairs:
        fa, fb = f1_matrix[a], f1_matrix[b]
        diff = fa - fb
        n = len(diff)

        # --- Shapiro-Wilk on the paired differences ---
        if n >= 3 and np.std(diff) > 0:
            try:
                _, p_norm = _scistats.shapiro(diff)
            except Exception:
                p_norm = 0.0
        else:
            p_norm = 0.0
        use_param = p_norm > alpha

        # --- Pick test + effect size ---
        if use_param:
            try:
                _, p_raw = _scistats.ttest_rel(fa, fb)
            except Exception:
                p_raw = 1.0
            sd = diff.std(ddof=1) if n > 1 else 0.0
            eff = float(diff.mean() / sd) if sd > 0 else 0.0
            test_used, eff_type = "paired_t", "cohens_d_z"
        else:
            if np.all(diff == 0):
                p_raw, eff = 1.0, 0.0
            else:
                try:
                    _, p_raw = _scistats.wilcoxon(fa, fb, zero_method='wilcox')
                except ValueError:
                    p_raw = 1.0
                eff = _rank_biserial(diff)
            test_used, eff_type = "wilcoxon", "rank_biserial"

        # --- 95% CI of the mean difference (t-interval) ---
        if n > 1 and diff.std(ddof=1) > 0:
            ci_lo, ci_hi = _scistats.t.interval(0.95, df=n - 1,
                                                loc=diff.mean(),
                                                scale=_scistats.sem(diff))
        else:
            ci_lo = ci_hi = float(diff.mean())

        if np.isnan(p_raw):
            p_raw = 1.0
        raw_pvals.append(float(p_raw))

        rows.append({
            'Dataset': dataset, 'Scenario': scenario, 'Run': _run_field,
            'Method_A': a, 'Method_B': b,
            'F1_A_pct':  round(float(fa.mean()) * 100, 4),
            'F1_B_pct':  round(float(fb.mean()) * 100, 4),
            'Delta_pct': round(float(fa.mean() - fb.mean()) * 100, 4),
            'N_seeds':   int(n),
            'Shapiro_p_diff': round(float(p_norm), 6),
            'Test_Used': test_used,
            'p_raw':     round(float(p_raw), 6),
            'Effect_Size': round(float(eff), 4),
            'Effect_Type': eff_type,
            'Magnitude':   _magnitude(eff, use_param),
            'CI95_low_diff_pct':  round(float(ci_lo) * 100, 4),
            'CI95_high_diff_pct': round(float(ci_hi) * 100, 4),
        })

    n_pairs = len(pairs)
    if n_pairs > 0:
        p_bonf = np.minimum(np.asarray(raw_pvals) * n_pairs, 1.0)
        p_bh = _bh_fdr(raw_pvals)
        for i, row in enumerate(rows):
            row['p_Bonferroni'] = round(float(p_bonf[i]), 6)
            row['p_BH_FDR']     = round(float(p_bh[i]),   6)
            row['Sig_Bonferroni'] = 'Sig' if p_bonf[i] < alpha else 'ns'
            row['Sig_BH_FDR']     = 'Sig' if p_bh[i]   < alpha else 'ns'

    sig_df = pd.DataFrame(rows)
    _mid = "_{}".format(run_label) if run_label else ""
    sig_path = os.path.join(
        output_dir,
        "significance_{}{}_{}.csv".format(dataset.lower(), _mid, scenario))
    sig_df.to_csv(sig_path, index=False)

    print("\n" + "=" * 72)
    _hdr = "SIGNIFICANCE TEST -- {} {} ({}) -- all-pairwise".format(dataset, scenario, run_label) if run_label else "SIGNIFICANCE TEST -- {} {} -- all-pairwise (full)".format(dataset, scenario)
    print(_hdr)
    print("=" * 72)
    print("  Methods:                {}".format(len(methods_list)))
    print("  Pairs tested:           {}".format(n_pairs))
    if n_pairs > 0:
        print("  Significant (Bonferroni, alpha={}): {}/{}".format(alpha, int((p_bonf < alpha).sum()), n_pairs))
        print("  Significant (BH-FDR,    alpha={}): {}/{}".format(alpha, int((p_bh   < alpha).sum()), n_pairs))
    print("  Saved to: {}".format(sig_path))
    print("=" * 72)
    return sig_df

In [ ]:
# =============================================
#   LOAD DATA — GANTI PATH SESUAI KAGGLE INPUT
# =============================================
print("="*70)
print(f"LOADING BC5CDR DATA — Scenario {FEW_SHOT_LABEL}")
print("="*70)

# GANTI PATH INI SESUAI LOKASI DATASET DI KAGGLE
DATASET_PATH = '/kaggle/input/datasets/nurhayatidbz85/bc5cdr'

train_sentences = parse_bc5cdr(os.path.join(DATASET_PATH, 'train.csv'))
val_sentences   = parse_bc5cdr(os.path.join(DATASET_PATH, 'validation.csv'))
test_sentences  = parse_bc5cdr(os.path.join(DATASET_PATH, 'test.csv'))

print(f"Train: {len(train_sentences)} sentences")
print(f"Val:   {len(val_sentences)} sentences")
print(f"Test:  {len(test_sentences)} sentences")

# Verify tags
all_tags_found = set()
for s in train_sentences:
    all_tags_found.update(s['tags'])
print(f"Tags found: {sorted(all_tags_found)}")

# Show sample
print(f"\nSample sentence:")
sample = train_sentences[0]
for tok, tag in zip(sample['tokens'][:10], sample['tags'][:10]):
    print(f"  {tok:20s} -> {tag}")

# [R1C7] Save split indices — exact sentences selected for few-shot training
fs_train = sample_few_shot(train_sentences, fs,
    save_path=os.path.join(OUTPUT_DIR, f'split_indices_{FEW_SHOT_LABEL}.json'))
n_chem = sum(1 for s in fs_train for t in s['tags'] if t == 'B-Chemical')
print(f"\nFew-shot {FEW_SHOT_LABEL}: {len(fs_train)} sentences")
print(f"  B-Chemical: {n_chem}")

# Compute adaptive IR threshold
IR_MEDIAN, IR_Q3 = compute_ir_stats(fs_train)
print(f"Adaptive IR: median={IR_MEDIAN:.2f}, Q3={IR_Q3:.2f}")


In [ ]:
# === CELL BREAK ===
print("\n" + "="*70)
print("BUILDING ENTITY EMBEDDINGS")
print("="*70)

mention_contexts, mention_types = extract_mentions_from_sentences(fs_train)
n_chem_mentions = sum(1 for v in mention_types.values() if v == 'Chemical')
print(f"Found {len(mention_contexts)} unique Chemical mentions")
embeddings = build_entity_embeddings(mention_contexts)
rankings = compute_similarity_rankings(embeddings, mention_types)
print("Done: Embeddings & rankings ready")

In [ ]:
# =============================================
#   RUN 1 — EXP 1-4
# =============================================
print("\n" + "="*70)
print(f"RUN 1: EXP 1-4 — BC5CDR {FEW_SHOT_LABEL}")
print("="*70)

results = {}
dataset_stats_run1 = []  # [R2C4][R2C5] For Table 8

# EXP 1: No Augmentation (Baseline)
print(f"\n--- EXP 1: No Augmentation (Baseline) ---")
dataset_stats_run1.append(compute_dataset_stats(fs_train, "No Augmentation", aug_count=0))
results["No Augmentation"] = run_multi_seed(fs_train, val_sentences, test_sentences, "NoAug")

# EXP 2: iBUS Only
print(f"\n--- EXP 2: iBUS Only (Rs={RS_DEFAULT}) ---")
ibus_data = ibus_undersample_dataset(fs_train, RS_DEFAULT,
    log_path=os.path.join(OUTPUT_DIR, f'ibus_filter_log_{FEW_SHOT_LABEL}.csv'))  # [R1C7]
dataset_stats_run1.append(compute_dataset_stats(ibus_data, "iBUS Only", aug_count=0))
results["iBUS Only"] = run_multi_seed(ibus_data, val_sentences, test_sentences, "iBUS")

# EXP 3: COSINER Only
print(f"\n--- EXP 3: COSINER Only ---")
aug_cos_only = run_cosiner(fs_train, rankings, imbalance_aware=False)
combined_cos_only = fs_train + aug_cos_only
print(f"    Combined: {len(combined_cos_only)} sentences | Aug instances: {len(aug_cos_only)}")
dataset_stats_run1.append(compute_dataset_stats(combined_cos_only, "COSINER Only", aug_count=len(aug_cos_only)))
results["COSINER Only"] = run_multi_seed(combined_cos_only, val_sentences, test_sentences, "COS-Only")

# EXP 4: iBUS then COSINER
print(f"\n--- EXP 4: iBUS then COSINER ---")
ibus_a = ibus_undersample_dataset(fs_train, RS_DEFAULT)
aug_a = run_cosiner(ibus_a, rankings, imbalance_aware=False)
combined_a = ibus_a + aug_a
print(f"    Combined: {len(combined_a)} sentences | Aug instances: {len(aug_a)}")
dataset_stats_run1.append(compute_dataset_stats(combined_a, "iBUS+COSINER", aug_count=len(aug_a)))
results["iBUS+COSINER"] = run_multi_seed(combined_a, val_sentences, test_sentences, "iBUS+COS")

In [ ]:
# # =============================================
# #   RUN 2 — EXP 5-8
# # =============================================
# print("\n" + "="*70)
# print(f"RUN 2: EXP 5-8 — BC5CDR {FEW_SHOT_LABEL}")
# print("="*70)

# results = {}
# # [R2C4][R2C5] Collect augmentation volume and training-set stats per configuration
# dataset_stats = []  # For Table 8 (R2C5)

# # EXP 5: COSINER then iBUS
# print(f"\n--- EXP 5: COSINER then iBUS ---")
# aug_b = run_cosiner(fs_train, rankings, imbalance_aware=False)
# combined_b_raw = fs_train + aug_b
# combined_b = ibus_undersample_dataset(combined_b_raw, RS_DEFAULT)
# print(f"    Combined: {len(combined_b)} sentences | Aug instances: {len(aug_b)}")
# dataset_stats.append(compute_dataset_stats(combined_b, "COSINER+iBUS", aug_count=len(aug_b)))
# results["COSINER+iBUS"] = run_multi_seed(combined_b, val_sentences, test_sentences, "COS+iBUS")

# # EXP 6: ADAMER Only
# print(f"\n--- EXP 6: ADAMER Only ---")
# aug_ia_only = run_cosiner(fs_train, rankings, imbalance_aware=True, ir_median=IR_MEDIAN, ir_q3=IR_Q3)
# combined_ia_only = fs_train + aug_ia_only
# print(f"    Combined: {len(combined_ia_only)} sentences | Aug instances: {len(aug_ia_only)}")
# dataset_stats.append(compute_dataset_stats(combined_ia_only, "ADAMER Only", aug_count=len(aug_ia_only)))
# results["ADAMER Only"] = run_multi_seed(combined_ia_only, val_sentences, test_sentences, "ADAMER")

# # EXP 7: ADAMER + iBUS (reversed order — ablation)
# print(f"\n--- EXP 7: ADAMER + iBUS (reversed order) ---")
# aug_ia = run_cosiner(fs_train, rankings, imbalance_aware=True, ir_median=IR_MEDIAN, ir_q3=IR_Q3)
# combined_ia_raw = fs_train + aug_ia
# combined_ia = ibus_undersample_dataset(combined_ia_raw, RS_DEFAULT)
# print(f"    Combined: {len(combined_ia)} sentences | Aug instances: {len(aug_ia)}")
# dataset_stats.append(compute_dataset_stats(combined_ia, "ADAMER+iBUS", aug_count=len(aug_ia)))
# results["ADAMER+iBUS"] = run_multi_seed(combined_ia, val_sentences, test_sentences, "ADAMER+iBUS")

# # EXP 8: iBUS + ADAMER (PROPOSED — full pipeline)
# print(f"\n--- EXP 8: iBUS + ADAMER (PROPOSED) ---")
# ibus_ia = ibus_undersample_dataset(fs_train, RS_DEFAULT)
# aug_ibus_ia = run_cosiner(ibus_ia, rankings, imbalance_aware=True, ir_median=IR_MEDIAN, ir_q3=IR_Q3)
# combined_ibus_ia = ibus_ia + aug_ibus_ia
# print(f"    Combined: {len(combined_ibus_ia)} sentences | Aug instances: {len(aug_ibus_ia)}")
# dataset_stats.append(compute_dataset_stats(combined_ibus_ia, "iBUS+ADAMER (Proposed)", aug_count=len(aug_ibus_ia)))
# # [R1C7] Save ADAMER augmentation output for reproducibility
# pd.DataFrame([{'id': s['id'], 'tokens': str(s['tokens']), 'tags': str(s['tags'])}
#               for s in aug_ibus_ia]).to_csv(
#     os.path.join(OUTPUT_DIR, f'adamer_aug_output_{FEW_SHOT_LABEL}.csv'), index=False)
# # [R2C6] Save per-token predictions of proposed method for failure mode analysis
# preds_path = os.path.join(OUTPUT_DIR, f'predictions_proposed_{FEW_SHOT_LABEL}')
# results["iBUS+ADAMER"] = run_multi_seed(combined_ibus_ia, val_sentences, test_sentences,
#                                          "iBUS+ADAMER", save_preds_path=preds_path)

# # [R2C5] Save training-set statistics for Table 8 in the paper — run2 EXPs only; merge with run1 if available
# stats_df = pd.DataFrame(dataset_stats)
# stats_df.insert(0, 'Dataset', 'BC5CDR')
# stats_df.insert(1, 'Few-shot', FEW_SHOT_LABEL)
# stats_df.to_csv(os.path.join(OUTPUT_DIR, f'dataset_stats_run2_{FEW_SHOT_LABEL}.csv'), index=False)
# print(f"\nDataset stats saved: dataset_stats_run2_{FEW_SHOT_LABEL}.csv")

In [ ]:
# =============================================
#   SAVING RESULTS — RUN1
# =============================================
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns, numpy as np, os

print("\n" + "="*70)
print("SAVING RESULTS — RUN1")
print("="*70)

rows = []
for method, m in results.items():
    rows.append({
        'Method': method, 'N_Train': m['n_train'],
        'F1':        f"{m['f1_mean']*100:.2f}% +/- {m['f1_std']:.3f}",
        'F1_CI95':   f"+/-{m['f1_ci95']:.3f}",
        'Precision': f"{m['p_mean']*100:.2f}% +/- {m['p_std']:.3f}",
        'Recall':    f"{m['r_mean']*100:.2f}% +/- {m['r_std']:.3f}",
        'F1_mean': round(m['f1_mean']*100, 4),
        'P_mean':  round(m['p_mean']*100, 4),
        'R_mean':  round(m['r_mean']*100, 4),
    })
df_results = pd.DataFrame(rows)
df_results.to_csv(os.path.join(OUTPUT_DIR, 'results_bc5cdr_run1_' + FEW_SHOT_LABEL + '.csv'), index=False)
print(f"\n=== BC5CDR {FEW_SHOT_LABEL} — RUN1 ===")
print(df_results[['Method', 'F1', 'Precision', 'Recall']].to_string(index=False))

seed_rows = []
for method, m in results.items():
    for i, seed in enumerate(SEEDS):
        seed_rows.append({
            'Dataset': 'BC5CDR', 'Scenario': FEW_SHOT_LABEL,
            'Run': 'run1', 'Method': method, 'Seed': seed,
            'F1': round(m['f1_scores'][i]*100, 4),
            'P':  round(m['p_scores'][i]*100, 4),
            'R':  round(m['r_scores'][i]*100, 4),
        })
pd.DataFrame(seed_rows).to_csv(
    os.path.join(OUTPUT_DIR, 'seed_details_bc5cdr_run1_' + FEW_SHOT_LABEL + '.csv'), index=False)

# Plot
methods = list(results.keys())
f1s  = [results[m]['f1_mean'] for m in methods]
stds = [results[m]['f1_std']  for m in methods]
colors = sns.color_palette("husl", len(methods))

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(methods, f1s, yerr=stds, capsize=5, color=colors,
              edgecolor='black', linewidth=0.5, alpha=0.85)
for bar, f1, std in zip(bars, f1s, stds):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + std + 0.005,
            f"{f1*100:.2f}%", ha='center', fontweight='bold', fontsize=9)
ax.set_ylabel('F1 Score (%)', fontsize=12)
ax.set_title('Method Comparison - BC5CDR (RUN1) (' + FEW_SHOT_LABEL + ')',
             fontsize=14, fontweight='bold')
y_max = max(f1s) + max(stds) + 0.08 if max(f1s) > 0 else 1.0
ax.set_ylim(0, y_max)
plt.xticks(rotation=25, ha='right'); ax.grid(axis='y', alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'comparison_bc5cdr_run1_' + FEW_SHOT_LABEL + '.png'),
            dpi=300, bbox_inches='tight')
plt.show(); plt.close()

best     = max(results.items(), key=lambda x: x[1]['f1_mean'])
baseline = results.get('No Augmentation', list(results.values())[0])
delta    = best[1]['f1_mean'] - baseline['f1_mean']
print(f"\nBest: {best[0]} -> F1={best[1]['f1_mean']*100:.2f}% (Delta={delta*100:+.2f}%)")
print(f"Output: {OUTPUT_DIR}")
print("Done — RUN1!")

# [R2C5] Save training-set statistics run1 for Table 8
stats_df_run1 = pd.DataFrame(dataset_stats_run1)
stats_df_run1.insert(0, 'Dataset', 'BC5CDR')
stats_df_run1.insert(1, 'Few-shot', FEW_SHOT_LABEL)
stats_df_run1.to_csv(os.path.join(OUTPUT_DIR, f'dataset_stats_run1_{FEW_SHOT_LABEL}.csv'), index=False)
print(f"Dataset stats run1 saved: dataset_stats_run1_{FEW_SHOT_LABEL}.csv")

# --- Statistical significance test (RUN1, all-pairwise) ---
run_pairwise_significance(results, 'BC5CDR', FEW_SHOT_LABEL, 'run1', OUTPUT_DIR)

In [ ]:
# # =============================================
# #   SAVING RESULTS — RUN2
# # =============================================
# import pandas as pd, matplotlib.pyplot as plt, seaborn as sns, numpy as np, os

# print("\n" + "="*70)
# print("SAVING RESULTS — RUN2")
# print("="*70)

# rows = []
# for method, m in results.items():
#     rows.append({
#         'Method': method, 'N_Train': m['n_train'],
#         'F1':        f"{m['f1_mean']*100:.2f}% +/- {m['f1_std']:.3f}",
#         'F1_CI95':   f"+/-{m['f1_ci95']:.3f}",
#         'Precision': f"{m['p_mean']*100:.2f}% +/- {m['p_std']:.3f}",
#         'Recall':    f"{m['r_mean']*100:.2f}% +/- {m['r_std']:.3f}",
#         'F1_mean': round(m['f1_mean']*100, 4),
#         'P_mean':  round(m['p_mean']*100, 4),
#         'R_mean':  round(m['r_mean']*100, 4),
#     })
# df_results = pd.DataFrame(rows)
# df_results.to_csv(os.path.join(OUTPUT_DIR, 'results_bc5cdr_run2_' + FEW_SHOT_LABEL + '.csv'), index=False)
# print(f"\n=== BC5CDR {FEW_SHOT_LABEL} — RUN2 ===")
# print(df_results[['Method', 'F1', 'Precision', 'Recall']].to_string(index=False))

# seed_rows = []
# for method, m in results.items():
#     for i, seed in enumerate(SEEDS):
#         seed_rows.append({
#             'Dataset': 'BC5CDR', 'Scenario': FEW_SHOT_LABEL,
#             'Run': 'run2', 'Method': method, 'Seed': seed,
#             'F1': round(m['f1_scores'][i]*100, 4),
#             'P':  round(m['p_scores'][i]*100, 4),
#             'R':  round(m['r_scores'][i]*100, 4),
#         })
# pd.DataFrame(seed_rows).to_csv(
#     os.path.join(OUTPUT_DIR, 'seed_details_bc5cdr_run2_' + FEW_SHOT_LABEL + '.csv'), index=False)

# # Plot
# methods = list(results.keys())
# f1s  = [results[m]['f1_mean'] for m in methods]
# stds = [results[m]['f1_std']  for m in methods]
# colors = sns.color_palette("husl", len(methods))

# fig, ax = plt.subplots(figsize=(12, 6))
# bars = ax.bar(methods, f1s, yerr=stds, capsize=5, color=colors,
#               edgecolor='black', linewidth=0.5, alpha=0.85)
# for bar, f1, std in zip(bars, f1s, stds):
#     ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + std + 0.005,
#             f"{f1*100:.2f}%", ha='center', fontweight='bold', fontsize=9)
# ax.set_ylabel('F1 Score (%)', fontsize=12)
# ax.set_title('Method Comparison - BC5CDR (RUN2) (' + FEW_SHOT_LABEL + ')',
#              fontsize=14, fontweight='bold')
# y_max = max(f1s) + max(stds) + 0.08 if max(f1s) > 0 else 1.0
# ax.set_ylim(0, y_max)
# plt.xticks(rotation=25, ha='right'); ax.grid(axis='y', alpha=0.3); plt.tight_layout()
# plt.savefig(os.path.join(OUTPUT_DIR, 'comparison_bc5cdr_run2_' + FEW_SHOT_LABEL + '.png'),
#             dpi=300, bbox_inches='tight')
# plt.show(); plt.close()

# best     = max(results.items(), key=lambda x: x[1]['f1_mean'])
# baseline = results.get('No Augmentation', list(results.values())[0])
# delta    = best[1]['f1_mean'] - baseline['f1_mean']
# print(f"\nBest: {best[0]} -> F1={best[1]['f1_mean']*100:.2f}% (Delta={delta*100:+.2f}%)")
# print(f"Output: {OUTPUT_DIR}")
# print("Done — RUN2!")

# # --- Statistical significance test (RUN2, all-pairwise) ---
# run_pairwise_significance(results, 'BC5CDR', FEW_SHOT_LABEL, 'run2', OUTPUT_DIR)